In [1]:
PROJECT_DIR = "/content/drive/MyDrive/en-pa.txt"

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
en_path = f"{PROJECT_DIR}/clean.en"
pa_path = f"{PROJECT_DIR}/clean.pa"
sp_model_path = f"{PROJECT_DIR}/punjabi_en_bpe.model"

In [4]:
def load_parallel(en_path, pa_path):
    with open(en_path, "r", encoding="utf-8") as f:
        en_lines = [line.strip() for line in f]

    with open(pa_path,"r",encoding="utf-8") as f:
        pa_lines = [line.strip() for line in f]

    assert len(en_lines) == len(pa_lines)

    pairs = list(zip(en_lines, pa_lines))
    return pairs

In [5]:
train_pairs = load_parallel(en_path=en_path , pa_path=pa_path)
print(len(train_pairs))
print(train_pairs[0])

820989
('And Our Command is but a single (Act),- like the twinkling of an eye [21].', '(ਅਤੇ) ਸਾਡਾ (ਕਿਸੇ ਪ੍ਰਕਾਰ ਦਾ ਕੋਈ) ਗੁਣ (ਜਾਂ) ਔਗੁਣ ਨਹੀਂ ਵਿਚਾਰਿਆ (ਭਾਵ ਸਭ ਕੁਝ ਅਖੋਂ ਓਹਲੇ ਕਰ ਦਿਤਾ) ।')


In [6]:
import sentencepiece as spm

In [7]:
sp = spm.SentencePieceProcessor()
sp.load(sp_model_path)

True

In [8]:
print("Vocab size:", sp.vocab_size())

print(sp.encode("I love machine learning"))
print(sp.encode("ਮੈਨੂੰ ਮਸ਼ੀਨ ਲਰਨਿੰਗ ਪਸੰਦ ਹੈ"))

Vocab size: 32000
[67, 1409, 5128, 5655]
[360, 4390, 27950, 13931, 1491, 31]


In [9]:
UNK_ID = 0
PAD_ID = 1
BOS_ID = 2
EOS_ID = 3

print("UNK:", sp.unk_id())
print("PAD:", sp.pad_id())
print("BOS:", sp.bos_id())
print("EOS:", sp.eos_id())


UNK: 0
PAD: 1
BOS: 2
EOS: 3


In [10]:
def encode_pair(en_text, pa_text):

    src_ids = [BOS_ID]
    src_ids += sp.encode(en_text)
    src_ids += [EOS_ID]

    tgt_ids = [BOS_ID]
    tgt_ids += sp.encode(pa_text)
    tgt_ids += [EOS_ID]

    return src_ids, tgt_ids

In [11]:
src,tgt = encode_pair(*train_pairs[0])
print(src)
print(tgt)

[2, 653, 2696, 18405, 88, 449, 6, 3352, 190, 29920, 125, 1446, 29918, 632, 12, 768, 689, 3027, 45, 198, 5942, 1512, 2248, 15261, 3]
[2, 190, 4856, 29930, 2631, 190, 12780, 10742, 118, 444, 29930, 2512, 190, 3237, 29930, 1190, 12775, 143, 20535, 190, 834, 400, 385, 5901, 85, 25654, 59, 14026, 29930, 709, 3]


In [12]:
from torch.utils.data import Dataset

In [13]:
class TranslationDataset(Dataset):

    def __init__(self,pairs):
        self.pairs = pairs 

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self,idx):
        en_text, pa_text = self.pairs[idx]
        src_ids,tgt_ids = encode_pair(
            en_text,
            pa_text
        )
        return src_ids,tgt_ids

In [14]:
train_dataset = TranslationDataset(train_pairs)

In [15]:
print(len(train_dataset))

src,tgt = train_dataset[0]
print(src)
print(tgt)

820989
[2, 653, 2696, 18405, 88, 449, 6, 3352, 190, 29920, 125, 1446, 29918, 632, 12, 768, 689, 3027, 45, 198, 5942, 1512, 2248, 15261, 3]
[2, 190, 4856, 29930, 2631, 190, 12780, 10742, 118, 444, 29930, 2512, 190, 3237, 29930, 1190, 12775, 143, 20535, 190, 834, 400, 385, 5901, 85, 25654, 59, 14026, 29930, 709, 3]


In [16]:
import torch
from torch.nn.utils.rnn import pad_sequence

In [17]:
def collate_fn(batch):
    src_batch = []
    tgt_batch = []

    for src_ids, tgt_ids in batch:

        src_batch.append(
            torch.tensor(src_ids, dtype=torch.long)
        )
        tgt_batch.append(
            torch.tensor(tgt_ids, dtype=torch.long)
        )

    src_batch = pad_sequence(
        src_batch,
        batch_first = True,
        padding_value = PAD_ID
    )

    tgt_batch = pad_sequence(
        tgt_batch,
        batch_first = True,
        padding_value = PAD_ID
    )

    return src_batch, tgt_batch

In [22]:
sample = [train_dataset[i] for i in range(4)]
src_batch,tgt_batch = collate_fn(sample)

print(src_batch.shape)
print(tgt_batch.shape)

torch.Size([4, 25])
torch.Size([4, 31])


In [23]:
def padding_mask(tokens):
    return(tokens != PAD_ID)

In [26]:
def create_casual_mask(seq_len):

    return torch.triu(
        torch.ones(seq_len, seq_len),
        diagonal = 1
    ).bool()

In [27]:
mask = create_casual_mask(5)
print(mask)

tensor([[False,  True,  True,  True,  True],
        [False, False,  True,  True,  True],
        [False, False, False,  True,  True],
        [False, False, False, False,  True],
        [False, False, False, False, False]])


In [28]:
from torch.utils.data import DataLoader

In [29]:
train_loader = DataLoader(
    train_dataset,
    batch_size = 64,
    shuffle = True,
    collate_fn = collate_fn,
    num_workers= 2
)

In [30]:
src_batch, tgt_batch = next(iter(train_loader))

In [31]:
tgt_input = tgt_batch[:,:-1]
tgt_output = tgt_batch[:,1:]

In [32]:
sample_src = [x for x in src_batch[0].tolist() if x != PAD_ID]
sample_tgt = [x for x in tgt_batch[0].tolist() if x != PAD_ID]

print("English:")
print(sp.decode(sample_src[1:-1]))

print("\nPunjabi:")
print(sp.decode(sample_tgt[1:-1]))

English:
The sunrise time of Nemuro in Eastern Hokkaido is around 6:10.

Punjabi:
ਪੂਰਬੀ ਹੋਕਾਇਡੋ ਵਿੱਚ ਨੀਮੂਰੋ ਦਾ ਸੂਰਜ ਚੜ੍ਹਨ ਦਾ ਸਮਾਂ ਲਗਭਗ 6:10 ਵਜੇ ਹੈ.


In [33]:
src_padding_mask = padding_mask(src_batch)

In [39]:
tgt_padding_mask = padding_mask(tgt_input)

In [35]:
memory_key_padding_mask = src_padding_mask

In [36]:
tgt_casual_mask = create_casual_mask(tgt_input.size(1))

In [40]:
print("Source:", src_batch.shape)
print("Target:", tgt_batch.shape)

print("Decoder Input:", tgt_input.shape)
print("Decoder Output:", tgt_output.shape)

print("Source Padding Mask:",
      src_padding_mask.shape)

print("Target Padding Mask:",
      tgt_padding_mask.shape)

print("Causal Mask:",
      tgt_casual_mask.shape)

Source: torch.Size([64, 37])
Target: torch.Size([64, 41])
Decoder Input: torch.Size([64, 40])
Decoder Output: torch.Size([64, 40])
Source Padding Mask: torch.Size([64, 37])
Target Padding Mask: torch.Size([64, 40])
Causal Mask: torch.Size([40, 40])
